# Diabetes 30-Day Readmission Prediction using XGBoost and SHAP

## Business Objective
This notebook builds the predictive layer of the Diabetes Readmission Intelligence Platform.

The goal is to predict whether a diabetic patient is likely to be readmitted within 30 days after discharge. This supports future discharge planning by identifying high-risk patients before they leave the hospital.

## Modeling Direction
The main model direction is XGBoost with SHAP explainability, based on literature showing that XGBoost achieved the strongest AUC-ROC among compared models on the same UCI Diabetes dataset.

## Important Evaluation Note
Because only around 11% of patients are readmitted within 30 days, accuracy alone is not reliable. The model will be evaluated using recall, precision, F1-score, ROC-AUC, confusion matrix, and business interpretation.

## Target Variable
readmitted_30d = 1 if readmitted == "<30"  
readmitted_30d = 0 otherwise

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

# XGBoost
from xgboost import XGBClassifier

# Explainability
import shap

# Settings
pd.set_option("display.max_columns", None)

print("Libraries loaded successfully")

In [ ]:
df = pd.read_csv("/content/diabetic_data.csv")
print(df.shape)
df.head()

In [ ]:
print("Rows, Columns:", df.shape)

display(df.head())

display(df.info())

display(df.describe(include="all"))

In [ ]:
# Convert ? to actual missing values

df = df.replace('?', np.nan)

# Missing values summary

missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)

missing["missing_percent"] = (
    missing["missing_count"] / len(df) * 100
).round(2)

missing.head(20)

In [ ]:
for col in [
    'race',
    'gender',
    'age',
    'weight',
    'payer_code',
    'medical_specialty',
    'max_glu_serum',
    'A1Cresult',
    'readmitted'
]:
    print("\n" + "="*60)
    print(col)
    print(df[col].value_counts(dropna=False))

In [ ]:
print(df.shape)

for col in [
    'race','gender','age','weight',
    'payer_code','medical_specialty',
    'max_glu_serum','A1Cresult','readmitted'
]:
    print(col, df[col].count())

In [ ]:
print("Total rows:", len(df))

print("Unique patients:",
      df['patient_nbr'].nunique())

print("Duplicate encounters:",
      len(df) - df['patient_nbr'].nunique())

In [ ]:
print("Current df shape:", df.shape)

# If you still have original loaded dataframe name, check it
try:
    print("Original df_raw shape:", df_raw.shape)
except NameError:
    print("df_raw does not exist")

print(df['readmitted'].value_counts(dropna=False))

In [ ]:
for col in df.columns:
    unique_count = df[col].nunique(dropna=False)

    if unique_count <= 5:
        print(f"\n{col}")
        print("-" * 40)
        print(df[col].value_counts(dropna=False))

# Exploratory Data Analysis (EDA) - Key Findings

### 1. Zero-Variance / Constant Features (To Be Dropped)
* **Features:** `examide`, `citoglipton`
* **Finding:** Both columns have 100% `No` values across all 101,766 records.
* **Action:** These provide zero predictive power and will be removed from the dataset.

### 2. Extremely Rare Medications (Under Review)
* **Features:** `troglitazone` (3 Steady), `glimepiride-pioglitazone` (1 Steady), `metformin-pioglitazone` (1 Steady), `metformin-rosiglitazone` (2 Steady).
* **Finding:** These features have near-zero variance as the vast majority of patients do not take them.
* **Action:** Retain for now to align with literature standards. We will decide whether to drop or keep them based on baseline domain research.

### 3. High-Variance / High-Importance Features
* **Feature:** `insulin`
* **Finding:** Well-distributed across all categories (`No`: 47,383, `Steady`: 30,849, `Down`: 12,218, `Up`: 11,316).
* **Action:** This is a vital clinical feature. It is expected to be a top predictor and highly sensitive during SHAP feature importance analysis.

### 4. Structurally Meaningful Missing Data (Lab Results)
* **Features:** `A1Cresult` (84% NaN) and `max_glu_serum` (95% NaN).
* **Finding:** The `NaN` values do not represent missing or corrupted data; rather, they signify **"Not Tested"**. This lack of testing is a deliberate clinical decision and carries strong predictive signal.
* **Action:** Do **not** drop these columns. Treat `NaN` as a distinct category (e.g., `Not Tested`).

---

## Critical Questions for Next EDA Phase

1. **Identifier Exclusion:** Are identifier columns (`encounter_id`, `patient_nbr`) properly isolated to prevent artificial inflation of model accuracy?
2. **Data Leakage Check:** Do any features contain information from the future (post-discharge) that could cause data leakage?
3. **Patient Duplication:** How many duplicate patients (`patient_nbr` appearing multiple times) exist in the dataset, and how will we handle multiple encounters per patient?

In [ ]:
weight_pct = (
    df['weight']
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(2)
)

print(weight_pct)

In [ ]:
(
    df['discharge_disposition_id']
    .value_counts()
    .sort_index()
)

In [ ]:
ids = pd.read_csv("/content/IDS_mapping.csv")

print(ids.head(20))
print(ids.columns)

In [ ]:
exclude_discharge_ids = [11, 13, 14, 19, 20, 21]

before_rows = len(df)

df = df[~df["discharge_disposition_id"].isin(exclude_discharge_ids)].copy()

after_rows = len(df)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Rows removed:", before_rows - after_rows)
print("New shape:", df.shape)

In [ ]:
id_cols = ['encounter_id', 'patient_nbr']

for col in id_cols:
    print("\n" + "="*50)
    print(col)
    print("Unique values:", df[col].nunique())
    print("Total rows:", len(df))

### Patient Identifier Analysis

A notable finding from the `patient_nbr` column is that the dataset contains:

- **99,343 total encounters**
- **69,990 unique patients**

This means there are **29,353 additional encounters** beyond the number of unique patients.

#### Insight

This indicates that many patients appear in the dataset more than once, suggesting that the data captures **multiple hospital encounters for the same individual** rather than a single record per patient.

This is an important characteristic of the dataset because previous healthcare utilization and repeated admissions may be associated with a patient's likelihood of future readmission.

In [ ]:
df['readmitted'].value_counts()

(
    df['readmitted']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
df['readmitted_30d'] = (
    df['readmitted'] == '<30'
).astype(int)

df['readmitted_30d'].value_counts()

### Handling Class Imbalance idea for Future modeling

The target variable is highly imbalanced, with only **11.39%** of encounters belonging to the `<30` readmission class. Several strategies can be considered to address this issue:

- **Class Weighting**: Increase the penalty for misclassifying minority-class observations using methods such as XGBoost's `scale_pos_weight`.
- **Random Under-Sampling**: Reduce the size of the majority class to create a more balanced training set.
- **Random Over-Sampling**: Duplicate minority-class samples to improve class balance.
- **SMOTE (Synthetic Minority Over-sampling Technique)**: Generate synthetic minority-class examples instead of duplicating existing records.
- **Ensemble-Based Under-Sampling (EasyEnsemble)**: Train multiple models on different balanced subsets of the majority class while keeping all minority-class observations. Final predictions are combined using voting or probability averaging.
- **Balanced Random Forest**: Train Random Forest models using balanced bootstrap samples.
- **Cost-Sensitive Learning**: Assign higher misclassification costs to minority-class observations during model training.

Among these approaches, Ensemble-Based Under-Sampling (EasyEnsemble) is particularly attractive because it preserves information from the majority class while avoiding the creation of synthetic data.

In [ ]:
print("diag_1 unique:", df["diag_1"].nunique())
print("diag_2 unique:", df["diag_2"].nunique())
print("diag_3 unique:", df["diag_3"].nunique())

print("\nTop 20 diag_1 values:")
display(df["diag_1"].value_counts().head(20))

In [ ]:
def map_diagnosis(icd):
    if pd.isna(icd):
        return "Missing"

    icd = str(icd).strip()

    # V codes and E codes
    if icd.startswith("V") or icd.startswith("E"):
        return "Other"

    try:
        icd_num = float(icd)
    except:
        return "Other"

    if 140 <= icd_num < 240:
        return "Neoplasms"
    elif str(icd).startswith("250"):
        return "Diabetes"
    elif (390 <= icd_num < 460) or icd_num == 785:
        return "Circulatory"
    elif (460 <= icd_num < 520) or icd_num == 786:
        return "Respiratory"
    elif (520 <= icd_num < 580) or icd_num == 787:
        return "Digestive"
    elif 800 <= icd_num < 1000:
        return "Injury"
    elif 700 <= icd_num < 740:
        return "Musculoskeletal"
    elif (580 <= icd_num < 630) or icd_num == 788:
        return "Genitourinary"
    else:
        return "Other"


df["diag_1_group"] = df["diag_1"].apply(map_diagnosis)
df["diag_2_group"] = df["diag_2"].apply(map_diagnosis)
df["diag_3_group"] = df["diag_3"].apply(map_diagnosis)

print(df[["diag_1", "diag_1_group", "diag_2", "diag_2_group", "diag_3", "diag_3_group"]].head())

print("\nDiagnosis group distribution:")
print(df["diag_1_group"].value_counts())

### 6. Diagnosis Grouping

#### Motivation
The original diagnosis variables (`diag_1`, `diag_2`, `diag_3`) contained hundreds of unique ICD-9 codes:

- diag_1: 715 unique values
- diag_2: 747 unique values
- diag_3: 786 unique values

Using raw ICD codes would create excessive sparsity after one-hot encoding and reduce interpretability.

#### Approach
ICD codes were mapped into clinically meaningful disease categories following approaches commonly used in the diabetes readmission literature.

Categories included:

- Circulatory
- Respiratory
- Digestive
- Diabetes
- Injury
- Musculoskeletal
- Genitourinary
- Neoplasms
- Other
- Missing

#### Outcome
The transformation reduced hundreds of diagnosis codes into a small set of clinically interpretable groups while preserving medical meaning.

The largest primary diagnosis category was Circulatory Disease (29,681 encounters), followed by Other (17,509) and Respiratory Disease (13,934).

In [ ]:
plt.figure(figsize=(10, 5))

df["diag_1_group"].value_counts().sort_values().plot(kind="barh")

plt.title("Distribution of Primary Diagnosis Groups")
plt.xlabel("Number of Encounters")
plt.ylabel("Diagnosis Group")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, ["diag_1_group", "diag_2_group", "diag_3_group"]):
    df[col].value_counts().sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel("Number of Encounters")
    ax.set_ylabel("")

plt.tight_layout()
plt.show()

In [ ]:
specialty_counts = (
    df["medical_specialty"]
    .value_counts(dropna=False)
)

print(specialty_counts.head(25))

In [ ]:
plt.figure(figsize=(10,8))

(
    df["medical_specialty"]
    .value_counts()
    .head(20)
    .sort_values()
    .plot(kind="barh")
)

plt.title("Top 20 Medical Specialties")
plt.xlabel("Count")
plt.show()

In [ ]:
readmission_by_specialty = pd.crosstab(
    df["medical_specialty"].fillna("Missing"),
    df["readmitted_30d"],
    normalize="index"
) * 100

readmission_by_specialty = (
    readmission_by_specialty
    .sort_values(1, ascending=False)
)

print(readmission_by_specialty.head(20))

In [ ]:
specialty_analysis = pd.crosstab(
    df["medical_specialty"].fillna("Missing"),
    df["readmitted_30d"]
)

specialty_analysis["total"] = specialty_analysis.sum(axis=1)

specialty_analysis["readmission_rate"] = (
    specialty_analysis[1]
    / specialty_analysis["total"]
    * 100
)

specialty_analysis = specialty_analysis.sort_values(
    "total",
    ascending=False
)

print(specialty_analysis.head(25))

In [ ]:
age_readmission = (
    df.groupby("age")["readmitted_30d"]
      .mean()
      .mul(100)
      .round(2)
      .reset_index()
)

print(age_readmission)

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    age_readmission["age"],
    age_readmission["readmitted_30d"],
    marker="o"
)

plt.title("30-Day Readmission Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Readmission Rate (%)")
plt.xticks(rotation=45)

plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Create prior utilization total
df["prior_utilization_total"] = (
    df["number_outpatient"] +
    df["number_emergency"] +
    df["number_inpatient"]
)

# Create prior utilization group
df["prior_utilization_group"] = pd.cut(
    df["prior_utilization_total"],
    bins=[-1, 0, 2, 5, df["prior_utilization_total"].max()],
    labels=[
        "0 - No Prior Utilization",
        "1-2 - Low Prior Utilization",
        "3-5 - Medium Prior Utilization",
        "6+ - High Prior Utilization"
    ]
)

df[["number_outpatient", "number_emergency", "number_inpatient",
    "prior_utilization_total", "prior_utilization_group"]].head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the heatmap data
age_utilization_heatmap = pd.crosstab(
    index=df["prior_utilization_group"],
    columns=df["age"],
    values=df["readmitted_30d"],
    aggfunc="mean"
).mul(100).round(2)

# Create the visualization
plt.figure(figsize=(14, 7))
sns.heatmap(
    age_utilization_heatmap,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.5,
    cbar_kws={'label': 'Readmission Rate (%)'}
)

# Refine UI/Labels
plt.title("Heatmap: 30-Day Readmission Rate by Age and Prior Utilization", fontsize=16, pad=20)
plt.xlabel("Age Group", fontsize=12)
plt.ylabel("Prior Healthcare Utilization Group", fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

# Exploratory Data Analysis (EDA)

## Dataset Overview

### Original Dataset

- Total encounters: 101,766
- Total features: 50

---

## Target Definition

The objective of this project is to predict whether a diabetic patient will be readmitted within 30 days of discharge.

### Target Variable

python readmitted_30d = 1 if readmitted == "<30" readmitted_30d = 0 otherwise

---

# Data Cleaning

## Removal of Hospice and Expired Patients

Following the methodology adopted in the literature, encounters associated with hospice care or patient death were removed from the dataset.

### Removed Discharge Disposition IDs

| ID | Description |
|----|-------------|
| 11 | Expired |
| 13 | Hospice (home) |
| 14 | Hospice (medical facility) |
| 19 | Expired at home (Medicaid hospice) |
| 20 | Expired in medical facility (Medicaid hospice) |
| 21 | Expired, place unknown |

### Results

| Metric | Value |
|----------|----------|
| Rows before cleaning | 101,766 |
| Rows after cleaning | 99,343 |
| Rows removed | 2,423 |

### Rationale

Patients who died or entered hospice care cannot realistically experience a future hospital readmission event. Including these records would introduce noise into the prediction task.

---

# Identifier Analysis

## encounter_id

### Findings

- Unique values: 99,343
- Total rows: 99,343

### Interpretation

encounter_id is a unique encounter identifier and does not contain predictive clinical information.

### Decision

Remove before model training.

---

## patient_nbr

### Findings

- Unique patients: 69,990
- Total encounters: 99,343

### Interpretation

Many patients appear multiple times in the dataset, confirming that the data is recorded at the encounter level rather than the patient level.

### Decision

Remove before model training.

---

# Class Imbalance Assessment

## Target Distribution

| Class | Count |
|---------|---------:|
| 0 (Not readmitted within 30 days) | 88,029 |
| 1 (Readmitted within 30 days) | 11,314 |

### Class Proportions

- Negative class: 88.61%
- Positive class: 11.39%

### Key Observation

The target variable exhibits substantial class imbalance.

### Implications

Future modeling experiments may compare:

- Standard XGBoost
- XGBoost with class weighting (scale_pos_weight ≈ 7.78)
- EasyEnsemble / Ensemble-Based Under-Sampling

---

# Missing Value Investigation

## weight

### Findings

- Missing values: 96.86%

### Interpretation

The variable contains very little usable information due to the extremely high proportion of missing values.

### Current Status

Strong candidate for removal.

---

## A1Cresult

### Findings

- Missing values: approximately 84%

### Interpretation

Missing values likely indicate that an HbA1c test was not performed.

### Decision

- Keep feature.
- Treat missing values as a meaningful category ("Not Tested").

---

## max_glu_serum

### Findings

- Missing values: approximately 95%

### Interpretation

Missing values likely indicate that glucose testing was not performed.

### Decision

- Keep feature.
- Treat missing values as a meaningful category ("Not Tested").

---

## medical_specialty

### Findings

- Missing values: approximately 49%

### Interpretation

Previous studies have adopted different strategies:

- Some remove the feature entirely.
- Others encode missing values as a separate category.

EDA findings indicate that readmission rates vary noticeably across specialties.

### Current Decision

- Keep the feature temporarily.
- Convert missing values into a dedicated category.
- Group rare specialties into "Other" during feature engineering.

---

# Constant Features

The following variables contain no predictive information:

- examide
- citoglipton

### Findings

Both variables contain only a single value across all encounters.

### Decision

Remove before model training.

---

# Diagnosis Feature Engineering

## Problem

The raw diagnosis variables contain hundreds of ICD codes:

| Feature | Unique Codes |
|----------|------------:|
| diag_1 | 715 |
| diag_2 | 747 |
| diag_3 | 786 |

Direct one-hot encoding would create a highly sparse feature space and reduce model interpretability.

---

## Solution

Diagnosis codes were grouped into clinically meaningful categories:

- Circulatory
- Respiratory
- Digestive
- Diabetes
- Injury
- Musculoskeletal
- Genitourinary
- Neoplasms
- Other
- Missing

### Examples

| ICD Code | Category |
|-----------|-----------|
| 250.83 | Diabetes |
| 414 | Circulatory |
| 786 | Respiratory |

---

## Distribution of Primary Diagnoses (diag_1)

| Category | Encounters |
|------------|----------:|
| Circulatory | 29,681 |
| Other | 17,509 |
| Respiratory | 13,934 |
| Digestive | 9,333 |
| Diabetes | 8,661 |
| Injury | 6,853 |
| Musculoskeletal | 5,219 |
| Genitourinary | 5,002 |
| Neoplasms | 3,131 |

### Key Insight

Circulatory conditions represent the dominant diagnosis category, which is consistent with findings from the SQL analysis and Tableau dashboard.

---

# Prior Utilization Feature Engineering

## Feature Creation

python prior_utilization_total = (     number_outpatient +     number_emergency +     number_inpatient )

### Utilization Groups

| Group | Definition |
|---------|------------|
| 0 - No Prior Utilization | 0 visits |
| 1-2 - Low Prior Utilization | 1–2 visits |
| 3-5 - Medium Prior Utilization | 3–5 visits |
| 6+ - High Prior Utilization | 6 or more visits |

---

## Key Finding

Readmission risk increases consistently as prior healthcare utilization increases.

### Example: Patients Aged 70–80

| Utilization Group | Readmission Rate |
|-------------------|----------------:|
| No Utilization | 9.51% |
| Low Utilization | 13.42% |
| Medium Utilization | 16.26% |
| High Utilization | 25.54% |

### Important Insight

Age alone provides limited predictive power. However, combining age with prior healthcare utilization reveals substantially stronger risk patterns.

This suggests that interaction effects may play an important role in predicting readmission risk. Tree-based algorithms such as XGBoost can often learn these interactions automatically without explicit feature engineering.

In [ ]:
diag_readmission = (
    df.groupby("diag_1_group")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

diag_readmission["readmission_rate"] = (
    diag_readmission["readmission_rate"] * 100
).round(2)

diag_readmission.sort_values(
    "readmission_rate",
    ascending=False
)

### Diagnosis Group vs Readmission Risk

#### Initial Observation

The **Missing** category is excluded from interpretation at this stage, as it contains only 20 records and is not statistically reliable.

#### Overall Readmission Benchmark

The overall 30-day readmission rate in the dataset is:

**11.39%**

The table below compares each diagnosis group against this benchmark.

| Diagnosis Group | Readmission Rate (%) |
|----------------|---------------------|
| Diabetes | 13.10 |
| Injury | 12.40 |
| Circulatory | 11.69 |
| Other | 11.61 |
| Genitourinary | 11.04 |
| Neoplasms | 10.89 |
| Digestive | 10.82 |
| Respiratory | 10.06 |
| Musculoskeletal | 9.91 |

#### Key Findings

**Diabetes shows the strongest signal**

Patients whose primary diagnosis is diabetes have the highest readmission rate (**13.10%**), approximately **15% higher than the dataset average (11.39%)**. This suggests that diabetes-related admissions are associated with an increased likelihood of 30-day readmission.

**Injury appears unexpectedly high**

The Injury category shows a readmission rate of **12.40%**, exceeding even the Circulatory group. This was not initially expected and may reflect complications associated with diabetic foot conditions, falls, fractures, or other trauma-related events. At this stage, it is recorded as an observation requiring further investigation.

**Circulatory remains a high-priority group**

Previous Tableau analysis identified Circulatory conditions as the largest contributor to estimated penalty exposure. The current results reinforce this finding, with a readmission rate of **11.69%**, above the dataset average. This combination of high patient volume and elevated readmission risk makes the Circulatory group particularly important from both clinical and operational perspectives.

**Respiratory risk is lower than expected**

The Respiratory group records a readmission rate of **10.06%**, which falls below the overall average. This suggests a weaker association with short-term readmission than initially anticipated.

**Musculoskeletal has the lowest risk**

Musculoskeletal conditions exhibit the lowest readmission rate (**9.91%**). A possible explanation is that many of these admissions relate to orthopedic or musculoskeletal issues that may involve fewer chronic complications than conditions such as diabetes or cardiovascular disease.

#### Implication for Machine Learning

Based on the observed variation in readmission rates across diagnosis categories, **`diag_1_group` should be retained as a predictive feature**.

Reasons:

- Meaningful differences in readmission risk exist across diagnosis groups.
- Sample sizes are sufficiently large for reliable interpretation.
- Findings are consistent with previous SQL and Tableau analyses.
- The feature has potential predictive value for readmission modeling.

In [ ]:
diagnosis_complexity = (
    df.groupby("number_diagnoses")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

diagnosis_complexity["readmission_rate"] = (
    diagnosis_complexity["readmission_rate"] * 100
).round(2)

print(diagnosis_complexity.sort_values("number_diagnoses"))

In [ ]:
los_analysis = (
    df.groupby("time_in_hospital")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

los_analysis["readmission_rate"] = (
    los_analysis["readmission_rate"] * 100
).round(2)

print(los_analysis)

## EDA Insight — Number of Diagnoses

### Business Question
Does patient clinical complexity increase 30-day readmission risk?

### Findings

A clear positive relationship exists between the number of recorded diagnoses and readmission risk.

| Number of Diagnoses | Readmission Rate |
|---|---:|
| 1 | 5.94% |
| 3 | 7.40% |
| 5 | 9.21% |
| 7 | 10.92% |
| 9 | 12.76% |

Patients with nine recorded diagnoses experienced more than double the readmission risk compared with patients who had only one diagnosis.

### Business Interpretation

The number of diagnoses appears to act as a proxy for overall patient complexity and comorbidity burden.

Patients with multiple concurrent medical conditions may require more intensive discharge planning, follow-up care, and post-discharge monitoring.

### Feature Engineering Implication

`number_diagnoses` should be retained as an important predictive feature for model training.

The feature demonstrates a strong and clinically intuitive relationship with readmission risk.

## EDA Insight — Length of Stay (Time in Hospital)

### Business Question
Does a longer hospital stay indicate elevated readmission risk?

### Findings

Readmission risk increases steadily as hospital length of stay increases.

| Time in Hospital (Days) | Readmission Rate |
|---|---:|
| 1 | 8.39% |
| 4 | 11.99% |
| 7 | 13.11% |
| 8 | 14.59% |
| 10 | 14.81% |

Patients with extended hospital stays showed substantially higher readmission rates than patients discharged after short stays.

### Business Interpretation

Longer hospital stays may reflect:

- greater disease severity,
- more complex treatment requirements,
- higher comorbidity burden,
- or increased clinical instability.

These patients may require enhanced discharge planning and follow-up interventions.

### Feature Engineering Implication

`time_in_hospital` should be retained as a core predictive feature.

The feature shows a clear relationship with readmission risk and is strongly supported by previous readmission prediction literature.

In [ ]:
a1c_analysis = (
    df.groupby("A1Cresult")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

a1c_analysis["readmission_rate"] = (
    a1c_analysis["readmission_rate"] * 100
).round(2)

a1c_analysis.sort_values(
    "readmission_rate",
    ascending=False
)

## EDA Insight — HbA1c Results

### Business Question
Do HbA1c test results themselves indicate higher readmission risk?

### Findings

Among patients who received an HbA1c test, readmission rates were relatively similar across result categories.

| HbA1c Result | Readmission Rate |
|---|---:|
| >7 | 10.15% |
| >8 | 9.94% |
| Normal | 9.77% |

Differences between categories were small and did not show a strong risk gradient.

### Business Interpretation

The actual HbA1c result appears to provide limited standalone predictive power.

The more important signal may not be the test result itself, but whether the patient was tested at all.

Previous SQL analysis showed that patients who were not tested had higher readmission rates than tested patients.

### Feature Engineering Implication

Instead of relying only on `A1Cresult`, an additional binary feature should be created:

- `hba1c_tested`

This feature may capture operational and clinical assessment behavior that is not reflected by the laboratory result alone.

In [ ]:
df["hba1c_tested"] = np.where(
    df["A1Cresult"].isna(),
    "Not Tested",
    "Tested"
)

In [ ]:
hba1c_tested_analysis = (
    df.groupby("hba1c_tested")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

hba1c_tested_analysis["readmission_rate"] = (
    hba1c_tested_analysis["readmission_rate"] * 100
).round(2)

print(hba1c_tested_analysis)

## EDA Insight — HbA1c Testing Status

### Findings

Patients without a recorded HbA1c test showed higher readmission rates than patients who were tested.

| HbA1c Testing Status | Readmission Rate |
|---|---:|
| Not Tested | 11.68% |
| Tested | 9.94% |

### Interpretation

The result suggests that HbA1c testing status contains predictive information associated with 30-day readmission risk.

However, this analysis is observational and does not establish causality.

Patients who were tested may differ from untested patients in other important characteristics such as diagnosis profile, disease severity, hospital utilization history, or treatment pathway.

Therefore, the finding should be interpreted as an association rather than evidence that HbA1c testing directly reduces readmissions.

### Feature Engineering Implication

Retain:

`hba1c_tested`

as a predictive feature for machine learning models.

The feature captures information that may reflect clinical assessment practices, patient management differences, or underlying patient risk characteristics.

In [ ]:
medication_analysis = (
    df.groupby("num_medications")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

medication_analysis["readmission_rate"] = (
    medication_analysis["readmission_rate"] * 100
).round(2)

print(medication_analysis.sort_values("num_medications"))

In [ ]:
print(df["num_medications"].describe())

In [ ]:
import matplotlib.pyplot as plt

med_plot = medication_analysis[
    medication_analysis["total_encounters"] >= 100
]

plt.figure(figsize=(10,5))
plt.plot(
    med_plot["num_medications"],
    med_plot["readmission_rate"]
)

plt.xlabel("Number of Medications")
plt.ylabel("Readmission Rate (%)")
plt.title("Readmission Rate vs Number of Medications")
plt.grid(True)
plt.show()

In [ ]:
med_plot = medication_analysis[
    medication_analysis["total_encounters"] >= 100
]

print(med_plot)

## EDA Summary — Strong Predictive Signals Identified

Several features have demonstrated meaningful associations with 30-day readmission risk.

### Prior Healthcare Utilization

Patients with higher historical healthcare utilization showed substantially higher readmission rates.

### Number of Diagnoses

Readmission risk increased steadily as the number of recorded diagnoses increased.

Patients with nine diagnoses experienced more than twice the readmission risk of patients with only one diagnosis.

### Length of Stay

Longer hospital stays were associated with higher readmission risk.

Patients hospitalized for 8–10 days showed substantially higher readmission rates than patients discharged after 1–2 days.

### HbA1c Testing Status

Patients without a recorded HbA1c test exhibited higher readmission rates than tested patients.

This finding should be interpreted as an association rather than evidence of causality.

### Emerging Pattern

The strongest predictors identified so far appear to reflect:

- patient complexity,
- comorbidity burden,
- healthcare utilization history,
- and intensity of care.

These findings align closely with previous literature on diabetes readmission prediction and provide strong justification for retaining these variables in the machine learning pipeline.

## EDA Insight — Medication Burden

### Business Question

Does medication burden increase 30-day readmission risk?

### Findings

A strong positive relationship was observed between the number of prescribed medications and readmission risk.

| Number of Medications | Readmission Rate |
|---|---:|
| 1 | 4.23% |
| 10 | 10.09% |
| 20 | 13.35% |
| 29 | 14.88% |
| 35 | 17.30% |

Readmission risk increased consistently as medication count increased.

### Business Interpretation

Medication count appears to be a proxy for patient complexity, disease burden, and treatment intensity.

Patients receiving large numbers of medications may require more careful discharge planning, medication reconciliation, and post-discharge monitoring.

### Feature Engineering Implication

`num_medications` should be retained as a core predictive feature.

Among the variables examined so far, medication burden demonstrates one of the strongest relationships with 30-day readmission risk.

In [ ]:
inpatient_analysis = (
    df.groupby("number_inpatient")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

inpatient_analysis["readmission_rate"] = (
    inpatient_analysis["readmission_rate"] * 100
).round(2)

print(inpatient_analysis)

## EDA Insight — Prior Inpatient Utilization

### Business Question

Does previous inpatient utilization increase future readmission risk?

### Findings

A very strong positive relationship exists between prior inpatient admissions and 30-day readmission risk.

| Prior Inpatient Visits | Readmission Rate |
|---|---:|
| 0 | 8.59% |
| 1 | 13.25% |
| 2 | 17.93% |
| 3 | 21.03% |
| 4 | 24.14% |
| 5 | 31.98% |
| 8 | 46.21% |

Patients with multiple previous inpatient admissions exhibited dramatically higher readmission rates than patients with no prior inpatient utilization.

### Business Interpretation

Previous inpatient admissions appear to be one of the strongest indicators of future readmission risk.

Frequent inpatient utilization may reflect:

- chronic disease burden,
- unresolved clinical issues,
- repeated care transitions,
- or persistent health instability.

### Feature Engineering Implication

`number_inpatient` should be retained as a high-priority predictive feature.

Among all variables examined so far, prior inpatient utilization demonstrates one of the strongest associations with 30-day readmission risk.

### Strategic Insight

Historical healthcare utilization appears to be a more powerful predictor of readmission than demographic characteristics alone.

This finding aligns closely with prior literature, where previous admissions consistently emerged as one of the most important readmission predictors.

In [ ]:
emergency_analysis = (
    df.groupby("number_emergency")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

emergency_analysis["readmission_rate"] = (
    emergency_analysis["readmission_rate"] * 100
).round(2)

print(emergency_analysis)

In [ ]:
print(df["number_emergency"].describe())

## EDA Insight — Prior Emergency Utilization

### Business Question

Does previous emergency department utilization increase future readmission risk?

### Findings

A strong positive relationship was observed between prior emergency visits and 30-day readmission risk.

| Prior Emergency Visits | Readmission Rate |
|---|---:|
| 0 | 10.69% |
| 1 | 14.68% |
| 2 | 18.75% |
| 3 | 20.82% |
| 4 | 30.89% |

Readmission risk increased substantially as the number of previous emergency visits increased.

### Business Interpretation

Frequent emergency utilization may indicate unstable disease management, worsening clinical conditions, or inadequate outpatient follow-up.

Patients with repeated emergency visits appear substantially more vulnerable to future readmissions.

### Feature Engineering Implication

`number_emergency` should be retained as an important predictive feature.

The feature contributes to a broader healthcare-utilization profile that consistently demonstrates strong predictive signal throughout the dataset.

In [ ]:
outpatient_analysis = (
    df.groupby("number_outpatient")
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

outpatient_analysis["readmission_rate"] = (
    outpatient_analysis["readmission_rate"] * 100
).round(2)

print(outpatient_analysis)

## EDA Summary — Healthcare Utilization History

Three healthcare utilization variables were examined:

- number_inpatient
- number_emergency
- number_outpatient

### Prior Inpatient Visits

Prior inpatient utilization demonstrated the strongest relationship with readmission risk.

Readmission rates increased from:

- 8.59% (0 prior inpatient visits)

to

- 46.21% (8 prior inpatient visits)

This was one of the strongest signals identified in the entire EDA process.

### Prior Emergency Visits

Emergency utilization also showed a strong positive relationship with readmission risk.

Patients with multiple previous emergency visits experienced substantially higher readmission rates than patients with no prior emergency utilization.

### Prior Outpatient Visits

Outpatient utilization demonstrated a weaker but still meaningful association with readmission risk.

The relationship was less pronounced than inpatient or emergency utilization, suggesting that outpatient care may reflect both disease burden and ongoing disease management.

### Strategic Insight

Historical healthcare utilization consistently emerged as one of the strongest predictor families in the dataset.

The findings suggest that previous interactions with the healthcare system may be more informative than demographic characteristics alone when predicting future readmissions.

### Feature Engineering Implication

Retain:

- number_inpatient
- number_emergency
- number_outpatient
- prior_utilization_total
- prior_utilization_group

for subsequent machine learning modeling and feature importance evaluation.

In [ ]:
feature = "change"

analysis = (
    df.groupby(feature)
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

analysis["readmission_rate"] = (
    analysis["readmission_rate"] * 100
).round(2)

print(analysis.sort_values("readmission_rate", ascending=False))

In [ ]:
feature = "diabetesMed"

analysis = (
    df.groupby(feature)
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

analysis["readmission_rate"] = (
    analysis["readmission_rate"] * 100
).round(2)

print(analysis.sort_values("readmission_rate", ascending=False))

In [ ]:
feature = "insulin"

analysis = (
    df.groupby(feature)
      .agg(
          total_encounters=("readmitted_30d", "size"),
          readmissions=("readmitted_30d", "sum"),
          readmission_rate=("readmitted_30d", "mean")
      )
      .reset_index()
)

analysis["readmission_rate"] = (
    analysis["readmission_rate"] * 100
).round(2)

print(analysis.sort_values("readmission_rate", ascending=False))

## Treatment & Medication Management Features

### Business Question

Do diabetes treatment and medication-management variables contain useful signals for predicting 30-day readmission risk?

### Features Analyzed

- `change`
- `diabetesMed`
- `insulin`

---

### 1. Medication Change (`change`)

| Category | Readmission Rate |
|-----------|-----------:|
| Changed | 12.02% |
| No Change | 10.84% |

#### Insight

Patients whose diabetes medication was adjusted during hospitalization experienced a moderately higher readmission rate.

This likely reflects greater clinical instability or treatment complexity rather than a direct causal effect of medication adjustment.

#### Feature Decision

KEEP ✅

Weak-to-moderate predictive signal.

---

### 2. Diabetes Medication (`diabetesMed`)

| Category | Readmission Rate |
|-----------|-----------:|
| Yes | 11.84% |
| No | 9.87% |

#### Insight

Patients receiving diabetes medication showed a noticeably higher readmission rate.

This may indicate more active or severe diabetes management needs.

#### Feature Decision

KEEP ✅

Moderate predictive signal.

---

### 3. Insulin Status (`insulin`)

| Category | Readmission Rate |
|-----------|-----------:|
| Down | 14.22% |
| Up | 13.33% |
| Steady | 11.37% |
| No | 10.21% |

#### Insight

Insulin-related interventions showed the clearest signal among treatment variables.

Patients whose insulin dosage was increased or decreased had substantially higher readmission rates than patients with no insulin treatment.

This suggests insulin management may act as a proxy for disease severity, treatment complexity, or glycemic instability.

#### Feature Decision

KEEP ✅

Strongest treatment-related signal identified so far.

---

### Overall Conclusion

Among treatment-related variables:

1. `insulin` → strongest signal
2. `diabetesMed` → moderate signal
3. `change` → weaker but still useful signal

These variables should be retained for future XGBoost modeling and SHAP explainability analysis.

Important: All findings are observational associations and should not be interpreted as causal relationships.

In [ ]:
features = [
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id"
]

for feature in features:
    print("\n" + "="*80)
    print(feature)

    analysis = (
        df.groupby(feature)
          .agg(
              total_encounters=("readmitted_30d", "size"),
              readmissions=("readmitted_30d", "sum"),
              readmission_rate=("readmitted_30d", "mean")
          )
          .reset_index()
    )

    analysis["readmission_rate"] = (
        analysis["readmission_rate"] * 100
    ).round(2)

    display(
        analysis.sort_values(
            "readmission_rate",
            ascending=False
        )
    )

## Care Pathway Features

### Features Analyzed

- admission_type_id
- admission_source_id
- discharge_disposition_id

---

### Key Findings

#### admission_type_id

Different admission pathways showed measurable differences in readmission risk.

Emergency and urgent admission categories generally exhibited higher readmission rates than elective pathways.

Feature Decision: KEEP ✅

---

#### admission_source_id

Readmission risk varied according to the patient's source of admission.

Several high-risk groups were observed, although some categories contained very small sample sizes and should not be over-interpreted.

Feature Decision: KEEP ✅

---

#### discharge_disposition_id

Discharge destination produced the strongest signal among the care-pathway variables.

Certain discharge destinations showed substantially elevated readmission rates, suggesting that post-discharge support requirements and patient frailty may be important drivers of readmission risk.

Feature Decision: KEEP ✅

---

### Overall Conclusion

Care-pathway variables contain useful predictive information and complement the clinical complexity and prior-utilization features identified earlier.

Among the three variables, discharge disposition appears to contain the strongest predictive signal and should be retained for future XGBoost and SHAP analysis.

Important: Small categories with very low sample sizes should be grouped or treated carefully during feature engineering.

In [ ]:
admission_type_map = {
    1: "Emergency",
    2: "Urgent",
    3: "Elective",
    4: "Newborn",
    5: "Not Available",
    6: "Unknown",
    7: "Trauma Center",
    8: "Not Mapped"
}

df["admission_type_group"] = (
    df["admission_type_id"]
      .map(admission_type_map)
      .fillna("Other")
)

In [ ]:
feature = "admission_type_group"

analysis = (
    df.groupby(feature)
      .agg(
          total_encounters=("readmitted_30d","size"),
          readmissions=("readmitted_30d","sum"),
          readmission_rate=("readmitted_30d","mean")
      )
      .reset_index()
)

analysis["readmission_rate"] = (
    analysis["readmission_rate"] * 100
).round(2)

analysis.sort_values(
    "readmission_rate",
    ascending=False
)

In [ ]:
admission_source_map = {
    1: "Physician Referral",
    2: "Clinic Referral",
    3: "HMO Referral",
    4: "Transfer Hospital",
    5: "Transfer SNF",
    6: "Transfer Healthcare Facility",
    7: "Emergency Room",
    8: "Court/Law Enforcement",
    9: "Not Available",
    10: "Critical Access Hospital",
    20: "Not Mapped",
    22: "Same Facility Transfer",
    25: "Transfer ASC"
}

df["admission_source_group"] = (
    df["admission_source_id"]
      .map(admission_source_map)
      .fillna("Other")
)

In [ ]:
discharge_map = {
    1: "Home",
    2: "Short-Term Hospital",
    3: "SNF",
    4: "ICF",
    5: "Inpatient Institution",
    6: "Home Health",
    7: "Left AMA",
    8: "Home IV",
    22: "Rehab Facility",
    23: "Long-Term Care",
    24: "Medicaid Nursing Facility",
    25: "Not Mapped",
    27: "Federal Facility",
    28: "Psychiatric Hospital"
}

df["discharge_group"] = (
    df["discharge_disposition_id"]
      .map(discharge_map)
      .fillna("Other")
)

In [ ]:
features = [
    "admission_source_group",
    "discharge_group"
]

for feature in features:

    print("\n" + "="*80)
    print(feature)

    analysis = (
        df.groupby(feature)
          .agg(
              total_encounters=("readmitted_30d", "size"),
              readmissions=("readmitted_30d", "sum"),
              readmission_rate=("readmitted_30d", "mean")
          )
          .reset_index()
    )

    analysis["readmission_rate"] = (
        analysis["readmission_rate"] * 100
    ).round(2)

    display(
        analysis.sort_values(
            "readmission_rate",
            ascending=False
        )
    )

## Admission and Discharge Pathway Features

### Features Analyzed

- admission_type_group
- admission_source_group
- discharge_group

---

### Admission Type

Emergency admissions showed slightly higher readmission rates than urgent and elective admissions.

Emergency: 11.83%
Urgent: 11.35%
Elective: 10.49%

Feature Decision: KEEP ✅

Signal strength: Moderate.

---

### Admission Source

Patients admitted through the Emergency Room showed higher readmission rates than physician referrals and transfer pathways.

Emergency Room: 11.98%
Physician Referral: 10.70%

Feature Decision: KEEP ✅

Signal strength: Moderate.

---

### Discharge Destination

Discharge destination produced one of the strongest signals observed so far.

Selected results:

- Psychiatric Hospital: 36.69%
- Rehab Facility: 27.70%
- Inpatient Institution: 20.86%
- SNF: 14.66%
- Home: 9.30%

Interpretation:

Patients requiring post-discharge institutional care appear substantially more likely to experience 30-day readmission.

Discharge destination may act as a proxy for patient frailty, functional limitations, illness severity, and ongoing care needs.

Feature Decision: KEEP ✅

Signal strength: Strong.

---

### Overall Conclusion

Care-pathway variables contain useful predictive information.

Among them, discharge destination appears substantially more informative than admission type or admission source and is likely to become an important feature in future XGBoost and SHAP analysis.

In [ ]:
feature = "max_glu_serum"

analysis = (
    df.groupby(feature)
      .agg(
          total_encounters=("readmitted_30d","size"),
          readmissions=("readmitted_30d","sum"),
          readmission_rate=("readmitted_30d","mean")
      )
      .reset_index()
)

analysis["readmission_rate"] = (
    analysis["readmission_rate"] * 100
).round(2)

analysis.sort_values(
    "readmission_rate",
    ascending=False
)

In [ ]:
df["max_glu_serum"].fillna("Not Tested").value_counts()

## Maximum Serum Glucose (max_glu_serum)

### Business Question

Does serum glucose testing and glucose severity contain useful information for predicting 30-day readmission risk?

### Findings

Distribution:

- Not Tested: 94,191 encounters
- Normal: 2,545 encounters
- >200: 1,419 encounters
- >300: 1,188 encounters

Readmission Rates (tested patients only):

| Category | Readmission Rate |
|-----------|-----------:|
| >300 | 15.07% |
| >200 | 12.97% |
| Norm | 11.55% |

### Interpretation

Patients with elevated maximum serum glucose levels experienced higher readmission rates.

However, the most important observation is that approximately 95% of encounters did not receive a serum glucose test.

Testing status itself may therefore contain operational and clinical information regarding patient assessment and disease complexity.

### Feature Engineering Decision

Create:

- `max_glu_tested`
- `max_glu_high`

KEEP ✅

### Caution

The observed relationship is associative rather than causal. Higher glucose values likely reflect greater clinical severity rather than directly causing readmission.